# 02 — Feature Engineering
**Projet** : ObRail MSPR 2025-2026  
**Auteure** : Charlotte  
**Source** : `data/processed/routes_processed.csv`  
**Résultat** : `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  

**Objectif** : Construire le feature set final, encoder les variables catégorielles, normaliser les numériques, et produire les splits train/test utilisés par tous les scripts de modélisation.

## 0. Imports et configuration

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'is_underserved'

print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'routes_processed.csv', dtype={'days_of_week': str})
print(f'Shape : {df.shape}')
df.head()

## 2. Suppression des colonnes sources de la cible

`is_underserved` est construit à partir de la règle :  
`days_active <= 3 AND distance_km > 100`

Ces deux colonnes sont donc des **fuites de données** directes — le modèle apprendrait la règle de construction plutôt que des patterns réels. Elles sont supprimées du feature set.

`log_distance` est conservée avec un caveat documenté : elle encode partiellement le seuil `distance_km > 100` mais capture aussi une réalité opérationnelle légitime (les longues routes sont structurellement plus difficiles à desservir fréquemment). Sa contribution sera vérifiée via SHAP après entraînement.

In [ ]:
LEAKY_COLS = ['days_active', 'distance_km']

# Colonnes non utilisées comme features (métadonnées textuelles ou identifiants)
META_COLS = [
    'agency_name', 'route_id', 'route_type',
    'route_short_name', 'route_long_name',
    'days_of_week', 'is_night_train',
    'arrival_country',
    'service_type',       # remplacée par type_encoded
    'departure_country',  # sera encodée en one-hot ci-dessous
    'co2_per_pkm',        # remplacée par log_co2
    'emissions_co2',      # non retenue (redondante avec log_co2)
    'country_encoded',    # remplacé par one-hot
]

cols_to_drop = LEAKY_COLS + META_COLS
print(f'Colonnes supprimées : {cols_to_drop}')

## 3. Encodage one-hot de `departure_country`

Le pays de départ est une variable catégorielle nominale — il n'existe pas d'ordre naturel entre les pays. Un encodage ordinal (FR=0, DE=1...) introduirait une relation numérique artificielle. On utilise donc le **one-hot encoding** : une colonne binaire par pays.

Avec 34 pays distincts, cela ajoute 34 colonnes. `drop='first'` supprime une colonne pour éviter la multicolinéarité parfaite (problème surtout pour la régression logistique).

In [ ]:
country_dummies = pd.get_dummies(
    df['departure_country'],
    prefix='country',
    drop_first=True,
    dtype=int,
)

print(f'Colonnes one-hot créées : {country_dummies.shape[1]}')
print(country_dummies.columns.tolist())

## 4. Construction du feature set final

In [ ]:
# Features numériques retenues
NUMERIC_FEATURES = [
    'rail_modal_share',   # part modale ferroviaire du pays — signal structurel
    'type_encoded',       # 0=jour, 1=nuit — fort discriminant
    'is_international',   # route internationale — signal très fort (94.4% underserved)
    'log_distance',       # distance log-transformée — caveat documenté section 2
    'log_co2',            # empreinte carbone log-transformée
]

# Assemblage : numériques + one-hot pays
X = pd.concat(
    [df[NUMERIC_FEATURES], country_dummies],
    axis=1,
)
y = df[TARGET]

print(f'Feature set final : {X.shape[1]} colonnes, {X.shape[0]} lignes')
print(f'\nFeatures numériques : {NUMERIC_FEATURES}')
print(f'Features one-hot    : {country_dummies.shape[1]} colonnes pays')
print(f'\nValeurs manquantes  : {X.isnull().sum().sum()}')
X.head()

### 4.1 Tableau des variables retenues

Livrable explicitement demandé par le cahier des charges ObRail (Besoin 1).

In [ ]:
variables_table = pd.DataFrame([
    {
        'Variable': 'rail_modal_share',
        'Type': 'Numérique continu',
        'Source': 'Eurostat (carte pays)',
        'Justification': 'Part modale ferroviaire du pays de départ — proxy du niveau d\'investissement réseau',
        'Corrélation cible': 0.09,
    },
    {
        'Variable': 'type_encoded',
        'Type': 'Binaire',
        'Source': 'is_night_train (encodé)',
        'Justification': 'Trains de nuit sous-desservis à 75.4% vs 17.9% pour les trains de jour',
        'Corrélation cible': 0.23,
    },
    {
        'Variable': 'is_international',
        'Type': 'Binaire',
        'Source': 'Comparaison departure/arrival country',
        'Justification': 'Routes internationales sous-desservies à 94.4% vs 18.5% pour domestiques',
        'Corrélation cible': 0.20,
    },
    {
        'Variable': 'log_distance',
        'Type': 'Numérique continu',
        'Source': 'log(1 + distance_km)',
        'Justification': 'Distance opérationnelle — corrélation 0.48 dont partie liée au seuil 100km de la règle cible. Vérifiée via SHAP.',
        'Corrélation cible': 0.48,
    },
    {
        'Variable': 'log_co2',
        'Type': 'Numérique continu',
        'Source': 'log(1 + co2_per_pkm)',
        'Justification': 'Empreinte carbone par passager-km — indicateur de type de traction et d\'efficacité',
        'Corrélation cible': -0.01,
    },
    {
        'Variable': 'country_XX (×33)',
        'Type': 'Binaire (one-hot)',
        'Source': 'departure_country (encodé)',
        'Justification': 'Effet pays — disparités importantes observées en EDA (NL 37.6%, SI 1.7%)',
        'Corrélation cible': 'variable',
    },
])

print('Tableau des variables retenues :')
display(variables_table)

# Sauvegarde du tableau
variables_table.to_csv(PROCESSED_DIR / 'variables_retenues.csv', index=False)
print('\n✅ Sauvegardé → data/processed/variables_retenues.csv')

## 5. Split train/test

**Ratio choisi : 80/20**
- 80% entraînement (~20 160 routes) : suffisant pour que les modèles apprennent
- 20% test (~5 040 routes) : assez large pour une évaluation robuste
- La cross-validation (5-fold) sera utilisée lors du tuning — pas besoin d'un set de validation séparé

**`stratify=y`** : garantit que le ratio 80.6% / 19.4% est respecté dans les deux splits — indispensable avec un déséquilibre de classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'\nDistribution y_train :')
print(y_train.value_counts())
print(f'Taux underserved train : {y_train.mean()*100:.1f}%')
print(f'\nDistribution y_test :')
print(y_test.value_counts())
print(f'Taux underserved test  : {y_test.mean()*100:.1f}%')

## 6. Normalisation des features numériques

**Pourquoi normaliser ?**
- La régression logistique et le MLP sont sensibles à l'échelle des features — `log_distance` va de 0.2 à 8.9, `rail_modal_share` de 1.9 à 17.9. Sans normalisation, les features à grande échelle dominent.
- LightGBM et RandomForest n'en ont pas besoin (modèles à base d'arbres) mais la normalisation ne les affecte pas non plus.
- On normalise donc tout le feature set pour avoir un pipeline unifié.

**Important** : le scaler est entraîné **uniquement sur X_train** puis appliqué à X_test. Entraîner sur l'ensemble complet introduirait une fuite d'information du test vers le train.

In [ ]:
# Colonnes à normaliser — uniquement les numériques continues
# Les binaires (type_encoded, is_international, country_XX) ne sont pas normalisées
COLS_TO_SCALE = ['rail_modal_share', 'log_distance', 'log_co2']

scaler = StandardScaler()

X_train = X_train.copy()
X_test  = X_test.copy()

X_train[COLS_TO_SCALE] = scaler.fit_transform(X_train[COLS_TO_SCALE])
X_test[COLS_TO_SCALE]  = scaler.transform(X_test[COLS_TO_SCALE])

print('Statistiques après normalisation (X_train) :')
display(X_train[COLS_TO_SCALE].describe().round(4))

print('\nStatistiques X_test (appliqué avec le scaler du train) :')
display(X_test[COLS_TO_SCALE].describe().round(4))

## 7. Sauvegarde des splits et du scaler

In [ ]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR  / 'y_test.csv',  index=False)

joblib.dump(scaler, PROCESSED_DIR / 'scaler.joblib')

print('✅ Splits sauvegardés :')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   y_train : {y_train.shape}')
print(f'   y_test  : {y_test.shape}')
print('✅ Scaler sauvegardé → data/processed/scaler.joblib')

## 8. Récapitulatif

**Ce qui a été fait dans ce notebook**
- Suppression des colonnes sources de `is_underserved` (`days_active`, `distance_km`)
- Encodage one-hot de `departure_country` (33 colonnes, `drop_first=True`)
- Feature set final : 5 numériques + 33 one-hot = **38 features**
- Split stratifié 80/20 — ratio underserved préservé dans les deux splits
- Normalisation StandardScaler sur les 3 features continues (`rail_modal_share`, `log_distance`, `log_co2`) — scaler entraîné sur train uniquement
- Sauvegarde de `X_train`, `X_test`, `y_train`, `y_test`, `scaler.joblib`, `variables_retenues.csv`

**Prochaine étape** : `03_models.ipynb` — entraînement et comparaison des modèles candidats sur ces splits.